In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
#Data Set and DataLoader

transform = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Normalize((0.5,),(0.5))
    
    ]
    
)

trainset = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
testset = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

In [3]:
trainloader = DataLoader(trainset, batch_size=64, shuffle=True)
testloader = DataLoader(testset, batch_size=64)

In [4]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 28 -> 14

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)   # 14 -> 7
        )

        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = self.fc(x)
        return x


In [5]:
class RNN(nn.Module):
    def __init__(self, input_size=28, hidden_size=128, num_layers=1):
        super(RNN, self).__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, 10)

    def forward(self, x):
        x = x.squeeze(1)  # (batch, 28, 28)

        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)

        out, _ = self.rnn(x, h0)
        out = self.fc(out[:, -1, :])

        return out


In [10]:
class LSTMModel(nn.Module):
    def __init__(self, input_size=28, hidden_size=128, num_layers=2):
        super(LSTMModel, self).__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, 10)

    def forward(self, x):
        # x shape: (batch, 1, 28, 28)
        x = x.squeeze(1)  # -> (batch, 28, 28)

        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)

        out, _ = self.lstm(x, (h0, c0))

        # Take last time step
        out = self.fc(out[:, -1, :])

        return out


In [6]:
def train_model(model, trainloader, epochs=5):
    model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for images, labels in trainloader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)

            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss:.4f}")


In [7]:
def evaluate_model(model, testloader):
    model.eval()

    y_true = []
    y_pred = []

    with torch.no_grad():
        for images, labels in testloader:
            images = images.to(device)

            outputs = model(images)
            _, predicted = torch.max(outputs, 1)

            y_true.extend(labels.numpy())
            y_pred.extend(predicted.cpu().numpy())

    print("Accuracy:", accuracy_score(y_true, y_pred))
    print("Precision:", precision_score(y_true, y_pred, average='macro'))
    print("Recall:", recall_score(y_true, y_pred, average='macro'))
    print("Confusion Matrix:\n", confusion_matrix(y_true, y_pred))


In [8]:
rnn_model = RNN()

print("Training RNN...")
train_model(rnn_model, trainloader, epochs=5)

print("\nEvaluating RNN...")
evaluate_model(rnn_model, testloader)


Training RNN...
Epoch 1/5, Loss: 738.1300
Epoch 2/5, Loss: 356.7494
Epoch 3/5, Loss: 255.7986
Epoch 4/5, Loss: 209.0650
Epoch 5/5, Loss: 190.1828

Evaluating RNN...
Accuracy: 0.9367
Precision: 0.9368862096469259
Recall: 0.9351442824190306
Confusion Matrix:
 [[ 961    0    3    3    0    5    2    3    3    0]
 [   0 1093    8    1    0    6    1    2   21    3]
 [   5    1 1006    6    3    1    1    7    1    1]
 [   2    0   14  957    0   25    0    8    0    4]
 [   1    1   11    2  909    0    5    2    1   50]
 [   6    2    3   39    6  783    2    5    8   38]
 [  47    2   11    0   32   18  829    0   19    0]
 [   2    0   10    3    1    0    0 1009    2    1]
 [   0    4   29   16    3   13    2   18  882    7]
 [   3    3    1    5    6   21    0   30    2  938]]


In [9]:
cnn_model = CNN()

print("\nTraining CNN...")
train_model(cnn_model, trainloader, epochs=5)

print("\nEvaluating CNN...")
evaluate_model(cnn_model, testloader)



Training CNN...
Epoch 1/5, Loss: 150.1682
Epoch 2/5, Loss: 41.6033
Epoch 3/5, Loss: 28.5335
Epoch 4/5, Loss: 20.8970
Epoch 5/5, Loss: 15.5328

Evaluating CNN...
Accuracy: 0.9943
Precision: 0.9942570033033352
Recall: 0.9942122715397643
Confusion Matrix:
 [[ 977    0    0    0    0    0    1    0    1    1]
 [   0 1134    0    0    0    0    0    1    0    0]
 [   0    1 1028    0    1    0    0    1    1    0]
 [   0    0    1 1005    0    1    0    0    2    1]
 [   0    0    0    0  977    0    0    0    1    4]
 [   1    0    0    4    0  885    1    0    1    0]
 [   1    2    0    0    1    1  951    0    2    0]
 [   0    2    4    0    0    0    0 1021    1    0]
 [   1    0    2    0    0    3    0    0  968    0]
 [   0    0    0    0    5    3    0    3    1  997]]


In [11]:
lstm_model = LSTMModel()

print("Training LSTM...")
train_model(lstm_model, trainloader, epochs=5)

print("\nEvaluating LSTM...")
evaluate_model(lstm_model, testloader)



Training LSTM...
Epoch 1/5, Loss: 341.2946
Epoch 2/5, Loss: 83.5921
Epoch 3/5, Loss: 59.2019
Epoch 4/5, Loss: 47.5448
Epoch 5/5, Loss: 39.4602

Evaluating LSTM...
Accuracy: 0.9839
Precision: 0.9840781377650402
Recall: 0.983690663009075
Confusion Matrix:
 [[ 974    0    1    0    1    0    2    1    1    0]
 [   0 1128    1    0    2    0    0    0    4    0]
 [   0    2 1028    0    1    0    0    1    0    0]
 [   0    0    4 1000    0    1    0    3    2    0]
 [   0    0    1    0  969    0    4    1    1    6]
 [   1    0    0   15    0  870    3    0    1    2]
 [   2    5    0    0    4    3  940    0    4    0]
 [   0    5   17    0    0    0    0 1002    0    4]
 [   2    0    6    2    0    2    0    1  959    2]
 [   1    4    1    1   24    3    0    5    1  969]]
